#  2-Tier SAR Ship Classifier

**Tier 1: Binary Classifier (Cargo vs Not Cargo)**
**Tier 2: Multi-Class Classifier (Non-Cargo Ship Types)**

The class imbalance problem in all three datasets is that cargo ships dominate. A single multi-class model ends up biased toward predicting cargo. The fix is two separate ResNet-18 models in sequence:
- Tier 1 filters out cargo ships confidently using is_cargo column
- Tier 2 runs only on what Tier 1 says is not cargo, classifying the remaining ship types

Both tiers trained on 4 datasets: os1, os2, fusar, and a joint pool of all three combined.
Both tiers use ResNet-18 with layers 1 and 2 frozen.


## Configuration

In [ ]:
import os
# import wandb
import torch
import pandas as pd
import numpy as np
import zipfile
import requests
from pathlib import Path

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'

# ---------------------------------------------------------------------------
# Box download URLs â€” fill these in with your shared links
# Box direct download format: https://company.box.com/shared/static/<id>
# ---------------------------------------------------------------------------
BOX_URLS = {
    "os1": "https://virginia.box.com/shared/static/9tz0fm7kq6745qzgy22hia985uf6c3hz.zip",   # e.g. "https://company.box.com/shared/static/abc123"
    "os2": "https://virginia.box.com/shared/static/t5dt2l6dgc3oz8oeizj83e8938gx69yr.zip",
    "fs":  "https://virginia.box.com/shared/static/u1lmcljm3o75gseghv5acnd4417qrkn2.zip",
}

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
DATA_DIR     = PROJECT_ROOT / "data" / "classification"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def download_and_extract(key: str, out_dir: Path) -> None:
    """Download a zip from Box and extract it to out_dir."""
    url = BOX_URLS[key]
    if not url:
        print(f"[{key}] No URL provided, skipping.")
        return
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"[{key}] Already extracted, skipping.")
        return
    zip_path = DATA_DIR / f"{key}.zip"
    print(f"[{key}] Downloading...")
    dl_url = url + ("?dl=1" if "?" not in url else "&dl=1")
    with requests.get(dl_url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"[{key}] Extracting...")
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    zip_path.unlink()
    print(f"[{key}] Done -> {out_dir}")


# ---------------------------------------------------------------------------
# Download image folders
# ---------------------------------------------------------------------------
download_and_extract("os1", DATA_DIR / "os1")
download_and_extract("os2", DATA_DIR / "os2")
download_and_extract("fs",  DATA_DIR / "fs")

# ---------------------------------------------------------------------------
# Load CSVs (already in repo at data/classification/)
# ---------------------------------------------------------------------------
opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

# Resolve local image paths
opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

fusar.head()

# Hyperparameters â€” same as baseline
BATCH_SIZE  = 32
EPOCHS      = 15
LR          = 1e-4
RANDOM_SEED = 42

### Notes - Configuration

Same auto-detection setup as ClassificationPipeline. Same hyperparameters too so results stay comparable to the baseline. Models for this notebook get saved to models/2tier/ so they don't overwrite the baseline checkpoints.

Same WandB project as the baseline so everything shows up in one place and you can compare the two-tier results directly against the single model results.


## Imports and WandB Login

In [ ]:
# import wandb
import gc
import json
import zipfile as zf_mod

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

# wandb.login()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


### Notes - Imports

Exact same imports as ClassificationPipeline. Nothing new here, just needed the same stack of torch, torchvision, wandb, sklearn, and seaborn.


## Load Label CSVs

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/19jLMSzHChVLk-vVAg2muNN2OALzksWob"

# Local directory to download into â€” sits next to this notebook
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "classification"

# Download from Google Drive if CSVs are not already present
if not (DATA_DIR / "opensar1_labels.csv").exists():
    print("Downloading classification folder from Google Drive...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(DRIVE_FOLDER_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)
    print("Download complete.")
else:
    print(f"Data already present at {DATA_DIR}, skipping download.")

# Load all three label CSVs
opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

# Resolve image paths to absolute local paths
opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

# print stats
print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

fusar.head()


In [ ]:
# Build mapping from all unique labels across all three datasets
unique_labels = sorted(set(opensar1['label']).union(
                       set(opensar2['label'])).union(
                       set(fusar['label'])))

label_to_int = {label: i for i, label in enumerate(unique_labels)}

# Apply mapping independently
opensar1.loc[:, 'label_id'] = opensar1['label'].map(label_to_int)
opensar2.loc[:, 'label_id'] = opensar2['label'].map(label_to_int)
fusar.loc[:, 'label_id']    = fusar['label'].map(label_to_int)

print("Label mapping:", label_to_int)

# drop unknown values of the datasets
opensar1 = opensar1[opensar1['label'] != 'unknown']
opensar2 = opensar2[opensar2['label'] != 'unknown']
fusar    = fusar[fusar['label'] != 'unknown']

# length after creation
print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

### Notes - Load Label CSVs

Same loading logic as before. The key thing here is that the is_cargo column already exists in all three CSVs with 0 or 1. That's what drives the whole two-tier approach. Tier 1 trains directly on that column so we don't need to derive anything.

The printout shows how imbalanced things are per dataset between cargo and non-cargo. That number is exactly why a single multi-class model struggles the cargo samples are swamping everything else during training.


## Build Joint Dataset

In [ ]:
# Joint dataset pools all three together into one combined training set
joint = pd.concat([opensar1, opensar2, fusar], ignore_index=True)

print(f'Joint dataset: {len(joint)} rows')
print(f'  cargo: {joint["is_cargo"].sum()} | not cargo: {(joint["is_cargo"]==0).sum()}')
print(f'\nFull label distribution in joint:')
print(joint['label'].value_counts().to_string())


### Notes - Joint Dataset

The joint dataset is just all three concatenated into one. The joint model outperformed individual models on the test set so we are training on all 4 variations: os1, os2, fusar, and joint. This gives us 4 Tier 1 models and 4 Tier 2 models, 8 total, so we can see which combination actually performs best.

The imbalance in the joint set is even more visible here since you are stacking three already imbalanced datasets. That is exactly the problem Tier 1 is designed to fix.


## Tier 1 - Label Setup (Binary: Cargo vs Not Cargo)

In [ ]:
# Tier 1 is binary: is_cargo column directly gives us 0 or 1
# 0 = not cargo, 1 = cargo
TIER1_CLASS_NAMES = ['not_cargo', 'cargo']
TIER1_NUM_CLASSES = 2

# Each domain dataframe already has is_cargo as 0/1 â€” use it directly as label_id
for df in [opensar1, opensar2, fusar, joint]:
    df['t1_label_id'] = df['is_cargo'].astype(int)

print('Tier 1 label mapping: 0 = not_cargo, 1 = cargo')
print(f'Number of classes: {TIER1_NUM_CLASSES}')

for name, df in [('os1', opensar1), ('os2', opensar2), ('fusar', fusar), ('joint', joint)]:
    counts = df['t1_label_id'].value_counts().sort_index()
    print(f'  {name}: not_cargo={counts.get(0,0)} | cargo={counts.get(1,0)}')


### Notes - Tier 1 Label Setup

Tier 1 is a binary classification problem. We don't need to build a fancy label mapping here because is_cargo is already 0 or 1 in the CSV. Not cargo is 0, cargo is 1. That is the entire job of Tier 1, just learn to separate those two groups. - Garebear make sure this is accurate


## Tier 2 - Label Setup (Multi-Class: Non-Cargo Ship Types)

In [ ]:
# Tier 2 only sees non-cargo ships
# Filter each dataset to is_cargo == 0
os1_nc    = opensar1[opensar1['is_cargo'] == 0].reset_index(drop=True)
os2_nc    = opensar2[opensar2['is_cargo'] == 0].reset_index(drop=True)
fusar_nc  = fusar[fusar['is_cargo'] == 0].reset_index(drop=True)
joint_nc  = joint[joint['is_cargo'] == 0].reset_index(drop=True)

# Build label mapping from non-cargo labels only
nc_labels = (set(os1_nc['label'])
             .union(set(os2_nc['label']))
             .union(set(fusar_nc['label'])))

TIER2_LABEL_TO_INT = {label: i for i, label in enumerate(sorted(nc_labels))}
TIER2_INT_TO_LABEL = {v: k for k, v in TIER2_LABEL_TO_INT.items()}
TIER2_CLASS_NAMES  = [TIER2_INT_TO_LABEL[i] for i in range(len(TIER2_LABEL_TO_INT))]
TIER2_NUM_CLASSES  = len(TIER2_CLASS_NAMES)

print(f'Tier 2 label mapping: {TIER2_LABEL_TO_INT}')
print(f'Class names: {TIER2_CLASS_NAMES}')
print(f'Number of classes: {TIER2_NUM_CLASSES}')

for df in [os1_nc, os2_nc, fusar_nc, joint_nc]:
    df['t2_label_id'] = df['label'].map(TIER2_LABEL_TO_INT)

print(f'\nNon-cargo sample counts:')
for name, df in [('os1_nc', os1_nc), ('os2_nc', os2_nc), ('fusar_nc', fusar_nc), ('joint_nc', joint_nc)]:
    print(f'  {name}: {len(df)} rows')
    print(f'  {df["label"].value_counts().to_string()}')
    print()


### Notes - Tier 2 Label Setup

Tier 2 only trains on the non-cargo rows. We filter first and then build the label mapping from whatever is left. Cargo is completely gone from this stage, Tier 1 already handled it. The remaining classes are things like tanker, passenger, engineering, other depending on what is in the dataset after filtering.

The non-cargo subsets are going to be much smaller than the full datasets which is expected. The point is that now the class distribution within Tier 2 should be a lot more balanced than the original data was with cargo included. That should help the model actually learn to distinguish between these rarer ship types instead of ignoring them.


## Dataset Class and Transforms

In [ ]:
RESNET_MEAN = [0.485, 0.456, 0.406]
RESNET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15, fill=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD)
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD)
])


class SARDataset(Dataset):
    def __init__(self, dataframe, label_col, transform=None):
        """
        label_col: which column to use as the label ('t1_label_id' for Tier 1, 't2_label_id' for Tier 2)
        """
        self.df        = dataframe.reset_index(drop=True)
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = Image.open(row['path']).convert('L')
        except (FileNotFoundError, OSError) as e:
            raise RuntimeError(f'Could not load image at index {idx}: {row["path"]}') from e
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row[self.label_col], dtype=torch.long)


print('SARDataset and transforms defined.')
print('Note: pass label_col="t1_label_id" for Tier 1, "t2_label_id" for Tier 2')


### Notes - Dataset Class and Transforms

Same SARDataset class as ClassificationPipeline with one change: it now takes a label_col argument. This is because the same image rows need to work for both tiers but pull from different label columns. For Tier 1 you pass t1_label_id which is the binary cargo/not-cargo column. For Tier 2 you pass t2_label_id which is the multi-class non-cargo label.

Everything else is identical to before, same transforms, same augmentation strategy, same ImageNet normalization.


## Tier 1 - Splits and DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader


def build_loader(dataset, batch_size, shuffle, domain_name):
    nw = 0 if domain_name == 'fusar' else 2
    pin_memory = domain_name != 'fusar'
    persistent_workers = nw > 0
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=nw,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
    )


def make_test_split(df, label_col, random_seed=RANDOM_SEED):
    """Split off 20% test indices once, shared across tiers."""
    labels = df[label_col].values
    idx = np.arange(len(df))

    trainval_idx, test_idx = train_test_split(
        idx, test_size=0.20, stratify=labels, random_state=random_seed
    )
    return trainval_idx, test_idx


def make_splits(df, label_col, trainval_idx, test_idx, domain_name, random_seed=RANDOM_SEED):
    """Build baseline-style train/val/test subsets and loaders for one domain."""
    labels = df[label_col].values

    train_idx, val_idx = train_test_split(
        trainval_idx,
        test_size=0.10,
        stratify=labels[trainval_idx],
        random_state=random_seed,
    )

    full_train = SARDataset(df, label_col, transform=transform_train)
    full_eval = SARDataset(df, label_col, transform=transform_eval)

    result = {
        'train_dataset': Subset(full_train, train_idx),
        'val_dataset': Subset(full_eval, val_idx),
        'test_dataset': Subset(full_eval, test_idx),
        'train_idx': list(train_idx),
        'val_idx': list(val_idx),
        'test_idx': list(test_idx),
    }
    result['train_loader'] = build_loader(result['train_dataset'], BATCH_SIZE, True, domain_name)
    result['val_loader'] = build_loader(result['val_dataset'], BATCH_SIZE, False, domain_name)
    result['test_loader'] = build_loader(result['test_dataset'], BATCH_SIZE, False, domain_name)

    return result, len(train_idx), len(val_idx), len(test_idx)


# Add source columns
opensar1['source'] = 'os1'
opensar2['source'] = 'os2'
fusar['source'] = 'fusar'
os1_nc['source'] = 'os1'
os2_nc['source'] = 'os2'
fusar_nc['source'] = 'fusar'

# Step 1: Split each source independently
shared_splits = {}
for name, df in [('os1', opensar1), ('os2', opensar2), ('fusar', fusar)]:
    shared_splits[name] = make_test_split(df, 't1_label_id')

# Step 2: Build joint indices by offsetting, not by re-splitting
os1_len = len(opensar1)
os2_len = len(opensar2)

trainval_os1, test_os1 = shared_splits['os1']
trainval_os2, test_os2 = shared_splits['os2']
trainval_fusar, test_fusar = shared_splits['fusar']

joint_trainval_idx = (
    list(trainval_os1)
    + [i + os1_len for i in trainval_os2]
    + [i + os1_len + os2_len for i in trainval_fusar]
)
joint_test_idx = (
    list(test_os1)
    + [i + os1_len for i in test_os2]
    + [i + os1_len + os2_len for i in test_fusar]
)
shared_splits['joint'] = (joint_trainval_idx, joint_test_idx)

# Step 3: Build Tier 1 domains
tier1_domains = {}
for name, df in [('os1', opensar1), ('os2', opensar2), ('fusar', fusar)]:
    trainval_idx, test_idx = shared_splits[name]
    tier1_domains[name], n_train, n_val, n_test = make_splits(
        df, 't1_label_id', trainval_idx, test_idx, name
    )
    print(f'Tier 1 | {name}: train={n_train} | val={n_val} | test={n_test}')

trainval_idx, test_idx = shared_splits['joint']
tier1_domains['joint'], n_train, n_val, n_test = make_splits(
    joint, 't1_label_id', trainval_idx, test_idx, 'joint'
)
print(f'Tier 1 | joint: train={n_train} | val={n_val} | test={n_test}')

# Save tier-1 split indices in the same style as the baseline notebook
Path('misc').mkdir(exist_ok=True)
for name, domain in tier1_domains.items():
    np.savez(
        f'misc/{name}_tier1_indices.npz',
        train_idx=np.array(domain['train_idx']),
        val_idx=np.array(domain['val_idx']),
        test_idx=np.array(domain['test_idx']),
    )

# Sanity check: no leakage
for name in ['os1', 'os2', 'fusar', 'joint']:
    d = tier1_domains[name]
    train_set = set(d['train_idx'])
    val_set = set(d['val_idx'])
    test_set = set(d['test_idx'])
    print(
        f"{name} | train/test overlap: {len(train_set & test_set)} | "
        f"train/val overlap: {len(train_set & val_set)} | "
        f"test/val overlap: {len(test_set & val_set)}"
    )

joint_test_set = set(tier1_domains['joint']['test_idx'])
os1_test_in_joint = {i for i in joint_test_set if i < os1_len}
os1_test_direct = set(tier1_domains['os1']['test_idx'])
print(f'os1 test consistent in joint: {os1_test_in_joint == os1_test_direct}')

os2_test_in_joint = {i - os1_len for i in joint_test_set if os1_len <= i < os1_len + os2_len}
os2_test_direct = set(tier1_domains['os2']['test_idx'])
print(f'os2 test consistent in joint: {os2_test_in_joint == os2_test_direct}')


### Notes - Tier 1 Splits and DataLoaders

Same stratified 70/10/20 split as before, now wrapped into a reusable make_splits function since we need to run it for both tiers across 4 datasets each. Stratified on t1_label_id here so both cargo and not-cargo maintain their proportions in each split.

We run this for all 4 datasets: os1, os2, fusar, and joint. That gives us 4 independent Tier 1 binary classifiers to train and compare.


## Tier 2 - Splits and DataLoaders

In [ ]:
opensar1

In [ ]:
# Step 1: Build tier 2 splits for individual sources
shared_splits_nc = {}
for name, (full_df, nc_df) in [
    ('os1', (opensar1, os1_nc)),
    ('os2', (opensar2, os2_nc)),
    ('fusar', (fusar, fusar_nc)),
]:
    _, test_idx = shared_splits[name]
    tier1_test_set = set(test_idx)

    safe_idx = np.array([i for i in range(len(nc_df)) if i not in tier1_test_set])
    labels = nc_df['t2_label_id'].values

    safe_trainval, safe_test = train_test_split(
        safe_idx,
        test_size=0.20,
        stratify=labels[safe_idx],
        random_state=RANDOM_SEED,
    )
    shared_splits_nc[name] = (safe_trainval, safe_test)
    print(f'{name}: {len(safe_idx)} safe rows ({len(nc_df) - len(safe_idx)} excluded from tier1 test)')

# Step 2: Build joint tier 2 by offsetting, not by re-splitting
os1_nc_len = len(os1_nc)
os2_nc_len = len(os2_nc)

trainval_os1_nc, test_os1_nc = shared_splits_nc['os1']
trainval_os2_nc, test_os2_nc = shared_splits_nc['os2']
trainval_fusar_nc, test_fusar_nc = shared_splits_nc['fusar']

joint_nc_trainval_idx = (
    list(trainval_os1_nc)
    + [i + os1_nc_len for i in trainval_os2_nc]
    + [i + os1_nc_len + os2_nc_len for i in trainval_fusar_nc]
)
joint_nc_test_idx = (
    list(test_os1_nc)
    + [i + os1_nc_len for i in test_os2_nc]
    + [i + os1_nc_len + os2_nc_len for i in test_fusar_nc]
)
shared_splits_nc['joint'] = (joint_nc_trainval_idx, joint_nc_test_idx)

# Step 3: Build tier 2 domains
tier2_domains = {}
for name, df in [('os1', os1_nc), ('os2', os2_nc), ('fusar', fusar_nc)]:
    trainval_idx, test_idx = shared_splits_nc[name]
    tier2_domains[name], n_train, n_val, n_test = make_splits(
        df, 't2_label_id', trainval_idx, test_idx, name
    )
    print(f'Tier 2 | {name}: train={n_train} | val={n_val} | test={n_test}')

trainval_idx, test_idx = shared_splits_nc['joint']
tier2_domains['joint'], n_train, n_val, n_test = make_splits(
    joint_nc, 't2_label_id', trainval_idx, test_idx, 'joint'
)
print(f'Tier 2 | joint: train={n_train} | val={n_val} | test={n_test}')

for name, domain in tier2_domains.items():
    np.savez(
        f'misc/{name}_tier2_indices.npz',
        train_idx=np.array(domain['train_idx']),
        val_idx=np.array(domain['val_idx']),
        test_idx=np.array(domain['test_idx']),
    )

with zf_mod.ZipFile(Path('misc/2tier_indices.zip'), 'w', zf_mod.ZIP_DEFLATED) as zf_out:
    for tier in ['tier1', 'tier2']:
        for name in ['os1', 'os2', 'fusar', 'joint']:
            npz_path = Path(f'misc/{name}_{tier}_indices.npz')
            if npz_path.exists():
                zf_out.write(npz_path, npz_path.name)
                npz_path.unlink()

for name in ['os1', 'os2', 'fusar', 'joint']:
    d = tier2_domains[name]
    train_set = set(d['train_idx'])
    val_set = set(d['val_idx'])
    test_set = set(d['test_idx'])
    print(
        f"{name} | train/test: {len(train_set & test_set)} | "
        f"train/val: {len(train_set & val_set)} | "
        f"test/val: {len(test_set & val_set)}"
    )


### Notes - Tier 2 Splits and DataLoaders

Same make_splits function, now applied to the non-cargo filtered dataframes. These are smaller datasets since cargo is gone, which you will see in the sample counts. The label column is t2_label_id now which maps to the non-cargo ship types.

Because the non-cargo classes are more balanced with each other now, the stratified split here is more meaningful than it was for the full datasets where cargo dominated everything.


## Model Imports and Configurations

In [ ]:
import sys
sys.path.insert(0, str(Path('.').resolve()))

from models.resnet_attention import resnet_attention
from models.resnet_transfer import CNN_resnet_transfer
from models.swin_transfer import swin_transfer

print('Models loaded:', resnet_attention, CNN_resnet_transfer, swin_transfer)


def build_model_configs(num_classes):
    return {
        'bam': {
            'cls': resnet_attention,
            'kwargs': {'num_classes': num_classes},
            'epochs': 15,
            'lr': 1e-4,
            'use_adamw': False,
            'warmup': False,
        },
        'res': {
            'cls': CNN_resnet_transfer,
            'kwargs': {'num_classes': num_classes},
            'epochs': 15,
            'lr': 1e-4,
            'use_adamw': False,
            'warmup': False,
        },
        'vis': {
            'cls': swin_transfer,
            'kwargs': {'num_classes': num_classes},
            'epochs': 10,
            'lr': 5e-5,
            'use_adamw': True,
            'warmup': True,
        },
    }


def sanity_check(model, label, train_loader, num_batches=3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    model.train()
    for i, (inputs, labels) in enumerate(train_loader):
        if i >= num_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        print(f'[{label}] Batch {i+1}/{num_batches} | Loss: {loss.item():.4f}')

    model.cpu()


TIER1_MODEL_CONFIGS = build_model_configs(TIER1_NUM_CLASSES)
TIER2_MODEL_CONFIGS = build_model_configs(TIER2_NUM_CLASSES)

for tag, cfg in TIER1_MODEL_CONFIGS.items():
    sanity_check(cfg['cls'](**cfg['kwargs']), f'tier1_os1_{tag}', tier1_domains['os1']['train_loader'])
    gc.collect()
    torch.cuda.empty_cache()

for tag, cfg in TIER2_MODEL_CONFIGS.items():
    sanity_check(cfg['cls'](**cfg['kwargs']), f'tier2_os1_{tag}', tier2_domains['os1']['train_loader'])
    gc.collect()
    torch.cuda.empty_cache()


### Notes - Model Setup

This notebook now uses the same three-model setup as `BaselineTrain`: transfer ResNet (`res`), BAM attention ResNet (`bam`), and Swin Transformer (`vis`).

Both tiers reuse the same baseline-style configuration pattern, but `num_classes` changes by tier so Tier 1 stays binary while Tier 2 targets the non-cargo label space.


## Training

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        if self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


def free_cuda(model=None):
    if model is not None:
        model.cpu()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


In [ ]:
import time


def train_model(model, save_path, class_names, train_loader, val_loader, train_labels,
                epochs=15, lr=1e-4, use_adamw=False, warmup=False):
    model = model.to(device)
    print(device)
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
    print(next(model.parameters()).device)

    criterion = FocalLoss(gamma=2.0)

    optimizer_cls = AdamW if use_adamw else Adam
    weight_decay = 1e-2 if use_adamw else 1e-4
    optimizer = optimizer_cls(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay,
    )

    if warmup:
        warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=2)
        cosine_sched = CosineAnnealingLR(optimizer, T_max=epochs - 2, eta_min=1e-6)
        scheduler = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[2])
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for i, (inputs, labels) in enumerate(train_loader):
            t0 = time.time()
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            t1 = time.time()
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            t2 = time.time()

            if i == 0 and epoch == 0:
                print(f'Data->GPU: {t1-t0:.3f}s | Forward+backward: {t2-t1:.3f}s')

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        scheduler.step()

        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        del inputs, labels, outputs, loss, predicted

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / len(val_loader)
        val_acc = 100.0 * correct / total
        current_lr = scheduler.get_last_lr()[0]

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(
            f'Epoch {epoch+1}/{epochs} | '
            f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
            f'Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | LR: {current_lr:.2e}'
        )

    model.load_state_dict(torch.load(save_path, map_location=device))
    return model, history


### Notes - Training

The training loop now matches the baseline notebook more closely: focal loss, cosine scheduling, optional warmup for Swin, and AdamW where the baseline used it.

Run names and checkpoint names include both the tier and the architecture so the saved outputs stay organized.


## Train All Tier 1 Models (Binary Classifier)

In [ ]:
Path('weights').mkdir(exist_ok=True)

tier1_models = {}
tier1_histories = {}

for tag, cfg in TIER1_MODEL_CONFIGS.items():
    tier1_models[tag] = {}
    tier1_histories[tag] = {}

    print(f'\n{'='*60}')
    print(f'  Tier 1 Training: {tag.upper()}')
    print(f'{'='*60}')

    for domain, d in tier1_domains.items():
        free_cuda()
        save_path = Path(f'weights/tier1_{domain}_{tag}.pth')
        train_labels = np.array(d['train_idx'])

        print(f'\n--- tier1 / {tag} / {domain} ---')
        model = cfg['cls'](**cfg['kwargs'])
        trained_model, history = train_model(
            model,
            save_path,
            TIER1_CLASS_NAMES,
            d['train_loader'],
            d['val_loader'],
            train_labels=train_labels,
            epochs=cfg['epochs'],
            lr=cfg['lr'],
            use_adamw=cfg['use_adamw'],
            warmup=cfg['warmup'],
        )

        tier1_models[tag][domain] = trained_model
        tier1_histories[tag][domain] = history



### Notes - Train All Tier 1 Models

This trains 4 independent binary classifiers, one per dataset. Each one is only trying to answer: is this a cargo ship or not. Because it is a binary problem the model has a much easier job than the original 5-class model and should converge faster and hit higher accuracy.

The joint model here is the most important one for the final pipeline since it has seen the most variety of SAR imagery across sensors. The per-dataset models are useful for comparison and understanding how sensor-specific the cargo/not-cargo distinction is.


## Train All Tier 2 Models (Multi-Class Non-Cargo Classifier)

In [ ]:
tier2_models = {}
tier2_histories = {}

for tag, cfg in TIER2_MODEL_CONFIGS.items():
    tier2_models[tag] = {}
    tier2_histories[tag] = {}

    print(f'\n{'='*60}')
    print(f'  Tier 2 Training: {tag.upper()}')
    print(f'{'='*60}')

    for domain, d in tier2_domains.items():
        free_cuda()
        save_path = Path(f'weights/tier2_{domain}_{tag}.pth')
        train_labels = np.array(d['train_idx'])

        print(f'\n--- tier2 / {tag} / {domain} ---')
        model = cfg['cls'](**cfg['kwargs'])
        trained_model, history = train_model(
            model,
            save_path,
            TIER2_CLASS_NAMES,
            d['train_loader'],
            d['val_loader'],
            train_labels=train_labels,
            epochs=cfg['epochs'],
            lr=cfg['lr'],
            use_adamw=cfg['use_adamw'],
            warmup=cfg['warmup'],
        )

        tier2_models[tag][domain] = trained_model
        tier2_histories[tag][domain] = history


### Notes - Train All Tier 2 Models

Tier 2 now mirrors Tier 1 exactly in structure, but each architecture is trained only on the non-cargo label space.

That means this notebook now trains 24 checkpoints total: 3 architectures x 4 domains x 2 tiers.


## Training Curves

In [ ]:
COLORS = {'os1': '#378ADD', 'os2': '#D4537E', 'fusar': '#1D9E75', 'joint': '#BA7517'}
png_paths = []

for tier_label, histories in [('tier1', tier1_histories), ('tier2', tier2_histories)]:
    for tag, domain_histories in histories.items():
        keys = list(domain_histories.keys())
        fig, axes = plt.subplots(len(keys), 2, figsize=(12, 4 * len(keys)))
        if len(keys) == 1:
            axes = np.array([axes])
        fig.suptitle(f'{tag.upper()} {tier_label.upper()} Training History', fontsize=14, y=1.01)

        for i, key in enumerate(keys):
            h = domain_histories[key]
            color = COLORS.get(key, 'steelblue')
            ep = range(1, len(h['train_loss']) + 1)

            ax1 = axes[i, 0]
            ax1.plot(ep, h['train_loss'], linestyle='dashed', color=color, label='train')
            ax1.plot(ep, h['val_loss'], linestyle='solid', color=color, label='val', linewidth=2)
            ax1.set_title(f'{key} - loss')
            ax1.set_xlabel('Epoch')
            ax1.set_ylabel('Loss')
            ax1.legend()

            ax2 = axes[i, 1]
            ax2.plot(ep, h['train_acc'], linestyle='dashed', color=color, label='train')
            ax2.plot(ep, h['val_acc'], linestyle='solid', color=color, label='val', linewidth=2)
            ax2.set_title(f'{key} - accuracy')
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Accuracy (%)')
            ax2.legend()

        plt.tight_layout()
        png = Path(f'misc/{tag}_{tier_label}_history.png')
        plt.savefig(png, dpi=150, bbox_inches='tight')
        plt.show()
        png_paths.append(png)

history_tier = {
    'tier1': tier1_histories,
    'tier2': tier2_histories,
}

json_path = Path('misc/2tier_history.json')
with open(json_path, 'w') as f:
    json.dump(history_tier, f, indent=2)

history_zip = Path('misc/2tier_history.zip')
with zf_mod.ZipFile(history_zip, 'w', zf_mod.ZIP_DEFLATED) as zf_out:
    zf_out.write(json_path, json_path.name)
    json_path.unlink()
    for png in png_paths:
        if png.exists():
            zf_out.write(png, png.name)
            png.unlink()

weights_zip = Path('weights/2tier_weights.zip')
with zf_mod.ZipFile(weights_zip, 'w', zf_mod.ZIP_DEFLATED) as zf_out:
    for tier in ['tier1', 'tier2']:
        for tag in ['bam', 'res', 'vis']:
            for domain in ['os1', 'os2', 'fusar', 'joint']:
                pth = Path(f'weights/{tier}_{domain}_{tag}.pth')
                if pth.exists():
                    zf_out.write(pth, pth.name)
                    pth.unlink()

print(f'History zipped to {history_zip}')
print(f'Weights zipped to {weights_zip}')


### Notes - Training Curves

The notebook now saves baseline-style curve artifacts for every architecture and both tiers, then bundles the plots and JSON history together in `misc/2tier_history.zip`.

Check the `weights/2tier_weights.zip` archive after training if you want all best checkpoints in one place.


## Test Set Evaluation

In [ ]:
def evaluate_test(model, run_name, test_loader, class_names):
    model = model.to(device)
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    model.cpu()
    torch.cuda.empty_cache()

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = 100.0 * (all_preds == all_labels).sum() / len(all_labels)

    print(f'\n{'='*50}')
    print(f'[{run_name}] Test Accuracy: {acc:.2f}% ({(all_preds == all_labels).sum()}/{len(all_labels)})')
    print(f'{'='*50}')
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    return {'accuracy': acc, 'preds': all_preds.tolist(), 'labels': all_labels.tolist()}


print('Running Tier 1 test evaluation...')
t1_results = {}
for tag, domain_models in tier1_models.items():
    t1_results[tag] = {}
    for model_name, model in domain_models.items():
        t1_results[tag][model_name] = {}
        for test_name, test_d in tier1_domains.items():
            run_name = f'tier1_{tag}_{model_name}_on_{test_name}'
            t1_results[tag][model_name][test_name] = evaluate_test(
                model, run_name, test_d['test_loader'], TIER1_CLASS_NAMES
            )

print('\nRunning Tier 2 test evaluation...')
t2_results = {}
for tag, domain_models in tier2_models.items():
    t2_results[tag] = {}
    for model_name, model in domain_models.items():
        t2_results[tag][model_name] = {}
        for test_name, test_d in tier2_domains.items():
            run_name = f'tier2_{tag}_{model_name}_on_{test_name}'
            t2_results[tag][model_name][test_name] = evaluate_test(
                model, run_name, test_d['test_loader'], TIER2_CLASS_NAMES
            )


In [ ]:
# Tier 1: 0=not_cargo, 1=cargo
CARGO_T2_IDX = 1
NOT_CARGO_T2_IDX = 0

ALL_CLASS_NAMES = sorted(TIER2_CLASS_NAMES + ['cargo'])
ALL_LABEL_TO_INT = {name: i for i, name in enumerate(ALL_CLASS_NAMES)}
CARGO_FULL_IDX = ALL_LABEL_TO_INT['cargo']
T2_TO_FULL = torch.tensor([ALL_LABEL_TO_INT[name] for name in TIER2_CLASS_NAMES], dtype=torch.long)

for df in [opensar1, opensar2, fusar, joint]:
    df['full_label_id'] = df['label'].map(ALL_LABEL_TO_INT)

hierarchical_test_domains = {}
for name, df in [('os1', opensar1), ('os2', opensar2), ('fusar', fusar), ('joint', joint)]:
    eval_dataset = SARDataset(df, 'full_label_id', transform=transform_eval)
    test_dataset = Subset(eval_dataset, tier1_domains[name]['test_idx'])
    hierarchical_test_domains[name] = {
        'test_dataset': test_dataset,
        'test_loader': build_loader(test_dataset, BATCH_SIZE, False, name),
        'test_idx': list(tier1_domains[name]['test_idx']),
    }

print(f'ALL_CLASS_NAMES:  {ALL_CLASS_NAMES}')
print(f'CARGO_FULL_IDX:   {CARGO_FULL_IDX}')
print(f'T2_TO_FULL:       {T2_TO_FULL.tolist()}')


def evaluate_hierarchical(tier1_model, tier2_model, run_name, test_loader):
    t2_to_full = T2_TO_FULL.to(device)

    tier1_model = tier1_model.to(device)
    tier2_model = tier2_model.to(device)
    tier1_model.eval()
    tier2_model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            batch_size = inputs.size(0)

            t1_preds = tier1_model(inputs).argmax(dim=1)
            final_preds = torch.full((batch_size,), CARGO_FULL_IDX, dtype=torch.long, device=device)

            non_cargo_mask = t1_preds == NOT_CARGO_T2_IDX
            if non_cargo_mask.any():
                t2_preds = tier2_model(inputs[non_cargo_mask]).argmax(dim=1)
                final_preds[non_cargo_mask] = t2_to_full[t2_preds]

            all_preds.extend(final_preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    tier1_model.cpu()
    tier2_model.cpu()
    torch.cuda.empty_cache()

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = 100.0 * (all_preds == all_labels).sum() / len(all_labels)

    print(f'\n{'='*50}')
    print(f'[{run_name}] Hierarchical Test Accuracy: {acc:.2f}%')
    print(f'  Cargo errors (tier1 misclassified cargo as non-cargo): {((all_labels == CARGO_FULL_IDX) & (all_preds != CARGO_FULL_IDX)).sum()}')
    print(f'  Non-cargo sent to cargo by tier1: {((all_labels != CARGO_FULL_IDX) & (all_preds == CARGO_FULL_IDX)).sum()}')
    print(f'{'='*50}')
    print(classification_report(all_labels, all_preds, target_names=ALL_CLASS_NAMES, zero_division=0))

    return {'accuracy': acc, 'preds': all_preds.tolist(), 'labels': all_labels.tolist()}


print('\nRunning hierarchical Tier 1 -> Tier 2 evaluation...')
hierarchical_results = {}
for tag in ['bam', 'res', 'vis']:
    hierarchical_results[tag] = {}
    for model_name in tier1_domains.keys():
        hierarchical_results[tag][model_name] = {}
        for test_name, test_d in hierarchical_test_domains.items():
            run_name = f'hier_{tag}_{model_name}_on_{test_name}'
            hierarchical_results[tag][model_name][test_name] = evaluate_hierarchical(
                tier1_models[tag][model_name],
                tier2_models[tag][model_name],
                run_name,
                test_d['test_loader'],
            )

test_results = {
    'tier1': t1_results,
    'tier2': t2_results,
    'hierarchical': hierarchical_results,
}

test_json_path = Path('misc/2tier_test_results.json')
with open(test_json_path, 'w') as f:
    json.dump(test_results, f, indent=2)

test_zip_path = Path('misc/2tier_test_results.zip')
with zf_mod.ZipFile(test_zip_path, 'w', zf_mod.ZIP_DEFLATED) as zf_out:
    zf_out.write(test_json_path, test_json_path.name)
    test_json_path.unlink()

print(f'Test results zipped to {test_zip_path}')


### Notes - Test Set Evaluation

This section now does all three evaluation passes automatically: Tier 1 alone, Tier 2 alone, and the full hierarchical Tier 1 to Tier 2 pipeline.

The hierarchical pass uses the same held-out test indices from Tier 1, rebuilds the full label space, and saves the combined results in `misc/2tier_test_results.zip`.


In [ ]:
model_colors = ['#7F77DD', '#1D9E75', '#D85A30', '#888780']

for tag in ['bam', 'res', 'vis']:
    domain_keys = list(tier1_domains.keys())
    x = np.arange(len(domain_keys))
    n = len(domain_keys)
    width = 0.8 / n

    fig, axes = plt.subplots(1, 3, figsize=(24, 5))

    ax = axes[0]
    for i, model_key in enumerate(domain_keys):
        accs = [t1_results[tag][model_key][t]['accuracy'] for t in domain_keys]
        offsets = x + i * width - (n - 1) * width / 2
        bars = ax.bar(offsets, accs, width, label=f'trained on {model_key}', color=model_colors[i], linewidth=0)
        for bar, acc in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8, f'{acc:.1f}', ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(domain_keys)
    ax.set_xlabel('Test set')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 105)
    ax.set_title(f'Tier 1 - {tag.upper()} Cross-domain Test Accuracy')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e0e0e0', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=9)

    ax = axes[1]
    for i, model_key in enumerate(domain_keys):
        accs = [t2_results[tag][model_key][t]['accuracy'] for t in domain_keys]
        offsets = x + i * width - (n - 1) * width / 2
        bars = ax.bar(offsets, accs, width, label=f'trained on {model_key}', color=model_colors[i], linewidth=0)
        for bar, acc in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8, f'{acc:.1f}', ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(domain_keys)
    ax.set_xlabel('Test set')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 105)
    ax.set_title(f'Tier 2 - {tag.upper()} Cross-domain Test Accuracy')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e0e0e0', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=9)

    ax = axes[2]
    for i, model_key in enumerate(domain_keys):
        accs = [hierarchical_results[tag][model_key][t]['accuracy'] for t in domain_keys]
        offsets = x + i * width - (n - 1) * width / 2
        bars = ax.bar(offsets, accs, width, label=f'trained on {model_key}', color=model_colors[i], linewidth=0)
        for bar, acc in zip(bars, accs):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8, f'{acc:.1f}', ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(domain_keys)
    ax.set_xlabel('Test set')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 105)
    ax.set_title(f'Hierarchical - {tag.upper()} Cross-domain Test Accuracy')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color='#e0e0e0', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join('misc', f'{tag}_cross_domain.png'), dpi=150)
    plt.show()


## Summary

In [ ]:
for tag in ['bam', 'res', 'vis']:
    print('\n' + '='*78)
    print(f'  {tag.upper()} - TIER 1 BINARY CLASSIFIER TEST RESULTS')
    print('='*78)
    print(f'{"Train Domain":<14}', ''.join(f'{t:>12}' for t in tier1_domains.keys()), f'{"Best Val":>10}')
    print('-'*78)
    for name in tier1_domains.keys():
        best_val = max(tier1_histories[tag][name]['val_acc'])
        test_accs = ''.join(f'{t1_results[tag][name][t]["accuracy"]:>11.2f}%' for t in tier1_domains.keys())
        print(f'{name:<14}{test_accs} {best_val:>9.2f}%')

    print('\n' + '='*78)
    print(f'  {tag.upper()} - TIER 2 MULTI-CLASS TEST RESULTS')
    print('='*78)
    print(f'{"Train Domain":<14}', ''.join(f'{t:>12}' for t in tier2_domains.keys()), f'{"Best Val":>10}')
    print('-'*78)
    for name in tier2_domains.keys():
        best_val = max(tier2_histories[tag][name]['val_acc'])
        test_accs = ''.join(f'{t2_results[tag][name][t]["accuracy"]:>11.2f}%' for t in tier2_domains.keys())
        print(f'{name:<14}{test_accs} {best_val:>9.2f}%')

    print('\n' + '='*78)
    print(f'  {tag.upper()} - HIERARCHICAL PIPELINE TEST RESULTS')
    print('='*78)
    print(f'{"Train Domain":<14}', ''.join(f'{t:>12}' for t in hierarchical_test_domains.keys()), f'{"Best Val":>10}')
    print('-'*78)
    for name in tier1_domains.keys():
        best_val = max(tier2_histories[tag][name]['val_acc'])
        test_accs = ''.join(f'{hierarchical_results[tag][name][t]["accuracy"]:>11.2f}%' for t in hierarchical_test_domains.keys())
        print(f'{name:<14}{test_accs} {best_val:>9.2f}%')


### Notes - Summary

The summary now prints three tables per architecture: Tier 1, Tier 2, and the full hierarchical pipeline.

If one architecture consistently wins on the `joint` row in the hierarchical table, that is probably the strongest candidate to carry forward into the final pipeline.
